In [33]:
import os
import time
import calendar

import numpy as np
import pandas as pd

import requests

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from selenium.common.exceptions import ElementNotInteractableException

from bs4 import BeautifulSoup as bs
from tqdm.notebook import tqdm

from bs4 import BeautifulSoup as bs

In [34]:
def udf_dt_gen(year, start=True):
    dates = []
    for month in range(1, 13):
        if start:
            day = 1
        else:
            # calendar.monthrange(year, month)[1]는 그 달의 마지막 날짜 반환
            day = calendar.monthrange(year, month)[1]
        date_str = f"{year}.{month:02d}.{day:02d}"
        dates.append(date_str)
    return dates

def sel_scroll_bottom(sleep = 0.3):
    remote.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(sleep)

sel_options = Options()
sel_options.add_argument("--headless")

In [35]:
val_search_text = "청년"
val_search_text_encoded = requests.utils.quote(val_search_text)
val_search_text_encoded

'%EC%B2%AD%EB%85%84'

In [62]:
ls_press_list = [[1, 2, 3],
                 ["hankook", "hankyoreh", "donga"],
                 [1469, 1028, 1020]] # 언론사 고유 번호

In [63]:
val_year_start = 2016
val_year_end   = 2025
val_time_sleep = 1

### Test 공간

In [39]:
bs_list_news[n_news].select("div.sds-comps-profile-source > div > .sds-comps-profile-info-subtext")[1].select_one('a')['href']
#url_bind

NameError: name 'bs_list_news' is not defined

In [32]:
bs_list_news[0].select('.sds-comps-base-layout > span')[0]

'2015.01.01.'

In [54]:
bs_list_news[n_news].select("div > div.sds-comps-vertical-layout > a > span")[1].text

'[한겨레] 위안부 합의·국정교과서 이후 맞은 3·1절 “정말 뭐라도 해야겠다 생각 들어” 청년들, 박근혜 대통령탈 등 제작 선언뒤 대현문화공원... 이들을 포함한 ‘한일 일본군 ‘위안부’ 합의 무효를 위한 대학생 대책위원회’ 소속 청년 400여명은 1일 서울 서대문구 대현문화공원에서 청계광장까지... '

### 뉴스 list 생성

In [ ]:
#for n_press in range(1, len(ls_press_list[0])):
for n_press in range(1):
    val_press_id = ls_press_list[0][n_press]
    val_press_nm = ls_press_list[1][n_press]
    val_press_cd = ls_press_list[2][n_press]
    # val_press_id, val_press_nm, val_press_cd
    print(val_press_nm)

    path_folder = f"../news_list/{val_press_nm}/"
    os.makedirs(path_folder, exist_ok = True)

    for val_year in tqdm(range(val_year_start, val_year_end + 1)):
        # val_year = 2015
        # print(val_year)
        # remote = webdriver.Chrome()
        remote = webdriver.Chrome(options = sel_options) # headless
        ls_date_s = udf_dt_gen(year = val_year)
        ls_date_e = udf_dt_gen(year = val_year, start = False)
    
        for n_month in range(12):
            # n_month = 0
            val_date_s = ls_date_s[n_month]
            val_date_e = ls_date_e[n_month]
            url_base = f"https://search.naver.com/search.naver?ssc=tab.news.all&query={val_search_text_encoded}&"
            url_param = f"sm=tab_opt&sort=2&photo=3&field=0&pd=3&ds={val_date_s}&de={val_date_e}&docid=&related=0&mynews=1&office_type=2&office_section_code=2&news_office_checked={val_press_cd}&is_sug_officeid=0&office_category=0&service_area=2" # photo=2 (only 지면기사)
            url_bind = url_base + url_param
        
            remote.get(url_bind)
            time.sleep(val_time_sleep)
        
            val_current_height = 0
            for n_scroll in range(30):
                # print(n_scroll)
                sel_scroll_bottom()
            
                val_new_height = remote.execute_script("return document.body.scrollHeight;")
                if val_new_height > val_current_height:
                    val_current_height = val_new_height
                    sel_scroll_bottom()
                else:
                    break
        
            res_source = remote.page_source
            bs_source = bs(res_source)
            
            bs_list_news = bs_source.select("ul.list_news div.desktop_mode > div > div > div")
            
            df_news_bind = pd.DataFrame()
            for n_news in range(len(bs_list_news)):
                # n_news = 0
                bs_news_profile = bs_list_news[n_news].select('div.sds-comps-profile-source > div > .sds-comps-profile-info-subtext')
                val_news_date = bs_news_profile[-2].text
                val_news_link = bs_news_profile[-1].select_one('a')['href']
                
                
                ls_news_id = val_news_link.split("/")[-2:]
                val_news_id = "/".join([ls_news_id[0], ls_news_id[1].split("?")[0]])
            
                bs_news_text = bs_list_news[n_news].select("div > div.sds-comps-vertical-layout > a > span")
                val_news_title = bs_news_text[0].text # bs_news_text[1]은 본문 일부
                val_news_summary = bs_news_text[1].text # bs_news_text[1]은 본문 일부
            
                df_news_single = pd.DataFrame(dict(date = [val_news_date],
                                                   id = [val_news_id],
                                                   link = [val_news_link],
                                                   title = [val_news_title],
                                                   summary = [val_news_summary]))
            
                df_news_bind = pd.concat([df_news_bind, df_news_single])
            
            df_news_bind = df_news_bind.reset_index(drop = True)
            
            val_file_ym = f"{val_year}{str(n_month + 1).zfill(2)}"
            path_file_write = f"{path_folder}/{val_file_ym}.tsv"
            df_news_bind.to_csv(path_file_write, index = False, sep = "\t")
        
        remote.quit()

### 뉴스 데이터 수집

In [66]:
ls_replace_main_text = [["\t", "\n", "\xa0", "&lt;", "&gt;", "<br/>", "<p>", "</p>"], 
                        [" ", "@", " ", "<", ">", "@", "@", "@"]]

In [67]:
val_time_sleep = 1

In [93]:
n_press = 2
val_press_id = ls_press_list[0][n_press]
val_press_nm = ls_press_list[1][n_press]
val_press_cd = ls_press_list[2][n_press]
val_press_id, val_press_nm, val_press_cd

(3, 'donga', 1020)

In [ ]:
path_folder_read = f"../news_list/{val_press_nm}/"
path_folder_write = f"../news_main/{val_press_nm}/"
os.makedirs(path_folder_write, exist_ok = True)

In [95]:
ls_file = os.listdir(path_folder_read)
df_bind = pd.DataFrame()
for n_file in tqdm(range(len(ls_file))):
    path_file = path_folder_read + ls_file[n_file]
    val_ym = ".".join([ls_file[n_file][:4], ls_file[n_file][4:6]])

    if os.path.getsize(path_file) < 5:
        df_sub = pd.DataFrame([[val_ym + ".01.", "", "", ""]],
                      columns = ["date", "id", "link", "title"])
    else:
        df_sub = pd.read_csv(path_file, sep = "\t")

    df_sub["ym"] = val_ym.replace(".", "")
    
    df_bind = pd.concat([df_bind, df_sub])

  0%|          | 0/120 [00:00<?, ?it/s]

In [96]:
df_bind = df_bind.loc[df_bind['link'].str.contains('n.news'),].drop_duplicates(subset='title')
df_bind.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13796 entries, 0 to 124
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   date     13796 non-null  object
 1   id       13796 non-null  object
 2   link     13796 non-null  object
 3   title    13796 non-null  object
 4   summary  13796 non-null  object
 5   ym       13796 non-null  object
dtypes: object(6)
memory usage: 754.5+ KB


In [97]:
# for n_ym in df_bind["ym"].unique()[-12:]:
for n_ym in df_bind['ym'].unique().tolist():
    # n_ym = "201404"
    time.sleep(2)
    print(n_ym)

    # remote = webdriver.Chrome()
    remote = webdriver.Chrome(options = sel_options) # headless
    df_ym_sub = df_bind.loc[df_bind["ym"] == n_ym, ].reset_index(drop = True)

    df_news_ym_bind = pd.DataFrame()
    for n_row in tqdm(range(len(df_ym_sub))):
        # print(n_row)
        # n_row = 101

        # 링크가 빈칸인거 예외처리 추가해야 함.
        
        val_news_id = df_ym_sub["id"].iloc[n_row]
        val_news_title = df_ym_sub["title"].iloc[n_row]
        url_news = df_ym_sub["link"].iloc[n_row]
    
        remote.get(url_news)
        time.sleep(val_time_sleep)
        sel_scroll_bottom()
    
        res_source = remote.page_source
        bs_source = bs(res_source)

        bs_news_timestamp = bs_source.select_one("div.media_end_head_info_datestamp_bunch > span")
        if bs_news_timestamp == None:
            try:
                val_news_timestamp = bs_source.select_one("div[class^=DateInfo] > em.date").text
            except:
                val_news_timestamp = np.nan
        else:
            val_news_timestamp = bs_news_timestamp.text
            
        bs_comment_count = bs_source.select_one("div.u_cbox_head > span.u_cbox_count")
        if bs_comment_count == None:
            # val_comment_count = bs_source.select_one("#cbox_module em.simplecmt_num").text
            val_comment_count = 0
        else:
            val_comment_count = bs_comment_count.text
        
        text_main = str(bs_source.select_one("#dic_area"))
        for n_repl in range(len(ls_replace_main_text[0])):
            text_main = text_main.replace(ls_replace_main_text[0][n_repl], 
                                          ls_replace_main_text[1][n_repl])
    
        df_news_single = pd.DataFrame(dict(id = [val_news_id],
                                           url = [url_news],
                                           timestamp = [val_news_timestamp],
                                           comment_count = [val_comment_count],
                                           title = [val_news_title],
                                           text = [text_main]))
        df_news_ym_bind = pd.concat([df_news_ym_bind, df_news_single])

    # remote.close()
    remote.quit()
    
    df_news_ym_bind = df_news_ym_bind.reset_index(drop = True)
    
    
    path_file_write = f"{path_folder_write}{n_ym}.tsv"
    df_news_ym_bind.to_csv(path_file_write, index = False, sep = "\t")

201601


  0%|          | 0/131 [00:00<?, ?it/s]

201602


  0%|          | 0/93 [00:00<?, ?it/s]

201603


...

  0%|          | 0/118 [00:00<?, ?it/s]

202511


  0%|          | 0/125 [00:00<?, ?it/s]

202512


  0%|          | 0/121 [00:00<?, ?it/s]

In [92]:
# for n_ym in df_bind["ym"].unique()[-12:]:
for n_ym in df_bind['ym'].unique():
    # n_ym = "201404"
    time.sleep(2)
    print(n_ym)

    # remote = webdriver.Chrome()
    remote = webdriver.Chrome(options = sel_options) # headless
    df_ym_sub = df_bind.loc[df_bind["ym"] == n_ym, ].reset_index(drop = True)

    df_news_ym_bind = pd.DataFrame()
    for n_row in tqdm(range(len(df_ym_sub))):
        # print(n_row)
        # n_row = 101

        # 링크가 빈칸인거 예외처리 추가해야 함.
        
        val_news_id = df_ym_sub["id"].iloc[n_row]
        val_news_title = df_ym_sub["title"].iloc[n_row]
        url_news = df_ym_sub["link"].iloc[n_row]
    
        remote.get(url_news)
        time.sleep(val_time_sleep)
        sel_scroll_bottom()
    
        res_source = remote.page_source
        bs_source = bs(res_source)

        bs_news_timestamp = bs_source.select_one("div.media_end_head_info_datestamp_bunch > span")
        if bs_news_timestamp == None:
            try:
                val_news_timestamp = bs_source.select_one("div[class^=DateInfo] > em.date").text
            except:
                val_news_timestamp = np.nan
        else:
            val_news_timestamp = bs_news_timestamp.text
            
        bs_comment_count = bs_source.select_one("div.u_cbox_head > span.u_cbox_count")
        if bs_comment_count == None:
            # val_comment_count = bs_source.select_one("#cbox_module em.simplecmt_num").text
            val_comment_count = 0
        else:
            val_comment_count = bs_comment_count.text
        
        text_main = str(bs_source.select_one("#dic_area"))
        for n_repl in range(len(ls_replace_main_text[0])):
            text_main = text_main.replace(ls_replace_main_text[0][n_repl], 
                                          ls_replace_main_text[1][n_repl])
    
        df_news_single = pd.DataFrame(dict(id = [val_news_id],
                                           url = [url_news],
                                           timestamp = [val_news_timestamp],
                                           comment_count = [val_comment_count],
                                           title = [val_news_title],
                                           text = [text_main]))
        df_news_ym_bind = pd.concat([df_news_ym_bind, df_news_single])

    # remote.close()
    remote.quit()
    
    df_news_ym_bind = df_news_ym_bind.reset_index(drop = True)
    
    
    path_file_write = f"{path_folder_write}{n_ym}.tsv"
    df_news_ym_bind.to_csv(path_file_write, index = False, sep = "\t")

201601


  0%|          | 0/176 [00:00<?, ?it/s]

201602


  0%|          | 0/101 [00:00<?, ?it/s]

201603


...

  0%|          | 0/66 [00:00<?, ?it/s]

202511


  0%|          | 0/50 [00:00<?, ?it/s]

202512


  0%|          | 0/62 [00:00<?, ?it/s]

In [ ]:
# remote.quit()

In [ ]:
df_news_ym_bind.head()

In [ ]:
# https://n.news.naver.com/mnews/article/comment/055/0000276295?sid=115
# 이걸로 댓글 접근 가능